# Визуализация диалогов: Мульти-модельная оценка

25-шаговый клинический encounter (PMC10783203) — врач vs ИИ-пациент

In [6]:
import json, glob, os
from IPython.display import HTML, display

result_files = sorted(glob.glob("results_*.json"))
print(f"Найдено файлов: {len(result_files)}")
for f in result_files:
    
    print(f"  {f}")

Найдено файлов: 3
  results_Claude_Sonnet_4.6.json
  results_Ministral_8B_OpenRouter.json
  results_Qwen_2.5_72B_OpenRouter.json


In [7]:
def render_dialogue(run_data, model_name):
    steps = run_data["steps"]
    n_passed = run_data["n_passed"]
    n_total = run_data["n_total"]
    first_fail = run_data["first_fail_step"]
    usage = run_data["usage"]
    run_num = run_data["run"]

    html = []
    html.append('<div style="color-scheme:light;color:#111827;max-width:900px;margin:20px auto;font-family:system-ui,sans-serif;">')

    # Header
    html.append(f'<div style="background:#1a1a2e;color:#eee;padding:16px 24px;border-radius:12px 12px 0 0;">')
    html.append(f'<h2 style="margin:0 0 8px 0;font-size:18px;">{model_name} — Прогон {run_num}</h2>')
    html.append(f'<span style="font-size:13px;color:#e2e8f0;">Результат: {n_passed}/{n_total} прошли | Первый сбой: шаг {first_fail} | Токены: in={usage["input_tokens"]}, out={usage["output_tokens"]}</span>')
    html.append(f'</div>')

    for s in steps:
        step_num = s["step"]
        phase = s["phase"]
        cls = s["class"]
        doctor = s["doctor"]
        patient = s["patient_response"]
        passed = s["passed"]
        fail_reason = s["fail_reason"]

        # Step header
        bg = "#c8e6c9" if passed else "#ffcdd2"
        border = "#4caf50" if passed else "#f44336"
        badge = "PASS" if passed else f"FAIL: {fail_reason}"
        badge_bg = "#4caf50" if passed else "#f44336"

        html.append(f'<div style="border-left:4px solid {border};background:{bg};margin:0;padding:12px 20px;">')
        html.append(f'<div style="display:flex;align-items:center;gap:12px;margin-bottom:8px;">')
        html.append(f'<span style="font-weight:700;font-size:14px;color:#111827;">Шаг {step_num}</span>')
        html.append(f'<span style="font-size:12px;color:#374151;">[{phase}]</span>')
        html.append(f'<span style="font-size:11px;background:#e0e0e0;color:#111827;padding:2px 8px;border-radius:10px;font-weight:600;">{cls}</span>')
        html.append(f'<span style="font-size:11px;background:{badge_bg};color:#fff;padding:2px 10px;border-radius:10px;font-weight:600;">{badge}</span>')
        html.append(f'</div>')

        # Doctor
        html.append(f'<div style="margin-bottom:8px;">')
        html.append(f'<span style="font-size:12px;font-weight:600;color:#1565c0;">Врач:</span> ')
        html.append(f'<span style="font-size:13px;color:#111827;">{doctor}</span>')
        html.append(f'</div>')

        # Patient
        html.append(f'<div style="background:#fff;border-radius:8px;padding:10px 14px;margin-left:20px;border:1px solid #9e9e9e;">')
        html.append(f'<span style="font-size:12px;font-weight:600;color:#e65100;">Пациент:</span> ')
        html.append(f'<span style="font-size:13px;color:#111827;">{patient}</span>')
        html.append(f'</div>')

        html.append(f'</div>')

    # Footer
    html.append(f'<div style="background:#1a1a2e;color:#e2e8f0;padding:12px 24px;border-radius:0 0 12px 12px;text-align:center;font-size:12px;">')
    html.append(f'Конец диалога — {model_name}, Прогон {run_num}')
    html.append(f'</div>')
    html.append(f'</div>')

    return "\n".join(html)


def render_comparison_table():
    """Сводная таблица всех моделей."""
    html = []
    html.append('<div style="color-scheme:light;color:#111827;max-width:900px;margin:20px auto;font-family:system-ui,sans-serif;">')
    html.append('<h2 style="margin:0 0 16px 0;font-size:18px;color:#111827;">Сравнительная таблица</h2>')
    html.append('<table style="width:100%;border-collapse:collapse;font-size:13px;border:1px solid #757575;">')
    html.append('<tr style="background:#1a1a2e;color:#fff;">')
    for th in ["Модель", "Прогон", "Прошли", "Первый сбой", "Input", "Output", "Стоимость"]:
        html.append(f'<th style="padding:10px 12px;text-align:left;border-bottom:2px solid #333;">{th}</th>')
    html.append('</tr>')

    pricing_map = {
        "Claude Sonnet 4.6": {"input": 3.0, "output": 15.0},
        "Qwen 2.5 72B (OpenRouter)": {"input": 0.65, "output": 0.65},
        "Ministral 8B (OpenRouter)": {"input": 0.1, "output": 0.1},
    }

    for fpath in sorted(glob.glob("results_*.json")):
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        model_name = data["model"]
        runs = data.get("results", [])
        for run in runs:
            if "error" in run:
                html.append(f'<tr style="color:#111827;background:#ffcdd2;border-bottom:1px solid #757575;"><td style="padding:8px 12px;">{model_name}</td><td style="padding:8px 12px;">{run["run"]}</td><td colspan="5" style="padding:8px 12px;color:#b71c1c;font-weight:600;">ОШИБКА: {run["error"][:80]}</td></tr>')
                continue
            usage = run["usage"]
            pricing = pricing_map.get(model_name, {"input": 0, "output": 0})
            cost = (usage["input_tokens"] * pricing["input"] + usage["output_tokens"] * pricing["output"]) / 1_000_000
            _full = run["n_passed"] == 25
            bg = "#c8e6c9" if _full else "#ffe0b2"
            _accent = "#2e7d32" if _full else "#e65100"
            html.append(f'<tr style="color:#111827;background:{bg};border-bottom:1px solid #9e9e9e;border-left:4px solid {_accent};">')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{model_name}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{run["run"]}</td>')
            html.append(f'<td style="padding:8px 12px;font-weight:700;color:#111827;">{run["n_passed"]}/25</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{run["first_fail_step"]}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{usage["input_tokens"]}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">{usage["output_tokens"]}</td>')
            html.append(f'<td style="padding:8px 12px;color:#111827;">${cost:.4f}</td>')
            html.append('</tr>')

    html.append('</table>')
    html.append('</div>')
    return "\n".join(html)

## Сравнительная таблица

In [8]:
display(HTML(render_comparison_table()))

Модель,Прогон,Прошли,Первый сбой,Input,Output,Стоимость
Claude Sonnet 4.6,1,22/25,1,0,36559,$0.5484
Claude Sonnet 4.6,2,22/25,1,0,47023,$0.7053
Claude Sonnet 4.6,3,24/25,1,0,36740,$0.5511
Ministral 8B (OpenRouter),1,25/25,Все прошли,35687,581,$0.0036
Ministral 8B (OpenRouter),2,25/25,Все прошли,35654,574,$0.0036
Ministral 8B (OpenRouter),3,25/25,Все прошли,36639,610,$0.0037
Qwen 2.5 72B (OpenRouter),1,23/25,1,41729,672,$0.0276
Qwen 2.5 72B (OpenRouter),2,23/25,1,42105,715,$0.0278
Qwen 2.5 72B (OpenRouter),3,23/25,1,41685,669,$0.0275


## Диалоги по моделям

Ниже выводятся **все прогоны** по **всем моделям** из `results_*.json` (успешные — полный HTML-чат; с ошибкой — строка в stdout). Между блоками — разделитель.

In [9]:
# Загружаем все результаты
all_data = {}
for fpath in sorted(glob.glob("results_*.json")):
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    all_data[data["model"]] = data

print("Доступные модели:")
for name, data in all_data.items():
    runs = data.get("results", [])
    ok = [r for r in runs if "error" not in r]
    print(f"  {name}: {len(ok)} прогонов")

Доступные модели:
  Claude Sonnet 4.6: 3 прогонов
  Ministral 8B (OpenRouter): 3 прогонов
  Qwen 2.5 72B (OpenRouter): 3 прогонов


In [12]:
for model_name in sorted(all_data.keys()):
    runs = all_data[model_name].get("results", [])
    for run in sorted(runs, key=lambda r: r.get("run", 0)):
        rn = run.get("run", "?")
        if "error" in run:
            print(f"[{model_name}] прогон {rn}: ошибка — {run['error'][:200]}")
            continue
        display(HTML(render_dialogue(run, model_name)))
        display(HTML('<hr style="margin:28px 0;border:none;border-top:2px solid #9e9e9e"/>'))
